<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/chatterbox-finetuning-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

## STEP 1: Configuration

Set up all configuration parameters from `src/config.py` along with source and destination paths.

In [ ]:
# STEP 1: Configuration

# Source and Destination Paths
source_path = '/content/drive/MyDrive/MyTTSDataset/'
dest_path = '/content/chatterbox-finetuning/'

# --- Parameters from src/config.py ---

# Path Settings
model_dir = "./pretrained_models"
csv_path = "./MyTTSDataset/metadata.csv"
wav_dir = "./MyTTSDataset/wavs"
preprocessed_dir = "./MyTTSDataset/preprocess"
output_dir = "./chatterbox_output"

# Inference Settings
is_inference = False
inference_prompt_path = "./speaker_reference/2.wav"
inference_test_text = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."

# Dataset Format Settings
ljspeech = True  # Set True if the dataset format is ljspeech, False if file-based
json_format = False  # Set True if dataset format is json, False if file-based or ljspeech

# Preprocessing mode: True (always run), False (skip), "auto" (smart detection)
preprocess = True

# Training Mode Settings
is_turbo = True  # True for Turbo training, False for Normal
is_lora = True   # True for LoRA training (recommended for < 10h data), False for Full Fine-Tune
is_merge_lora = True  # If True and is_lora is True, automatically merge LoRA weights after training

# LoRA Parameters
lora_r = 128
lora_alpha = 256
turbo_lora_target_modules = ["c_attn", "c_proj", "c_fc", "spkr_enc"]
lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "spkr_enc"]
lora_modules_to_save = ["text_emb", "text_head"]

# Vocabulary Settings
new_vocab_size = 52260 if is_turbo else 2454

# Hyperparameters
batch_size = 8
grad_accum = 4
learning_rate = 1e-4 if is_lora else 1e-5
num_epochs = 10 if is_lora else 30
save_steps = 500
save_total_limit = 5
dataloader_num_workers = 8

# Constraints
start_text_token = 255
stop_text_token = 0
max_text_len = 256
max_speech_len = 850
prompt_duration = 3.0

# Print configuration summary
print("=== Configuration Summary ===")
print(f"Source Path: {source_path}")
print(f"Destination Path: {dest_path}")
print(f"Model Dir: {model_dir}")
print(f"CSV Path: {csv_path}")
print(f"WAV Dir: {wav_dir}")
print(f"Preprocessed Dir: {preprocessed_dir}")
print(f"Output Dir: {output_dir}")
print(f"Is Turbo: {is_turbo}")
print(f"Is LoRA: {is_lora}")
print(f"Is Merge LoRA: {is_merge_lora}")
print(f"LoRA R: {lora_r}")
print(f"LoRA Alpha: {lora_alpha}")
print(f"Batch Size: {batch_size}")
print(f"Grad Accum: {grad_accum}")
print(f"Learning Rate: {learning_rate}")
print(f"Num Epochs: {num_epochs}")
print("==============================")

## STEP 2: Clone Repository & Sync Configuration

Clone the Chatterbox repository and sync the configuration parameters to `src/config.py`.

In [ ]:
# STEP 2: Clone Repository & Sync Configuration

import os

# Clone the repository (adjust the URL if needed)
repo_url = "https://github.com/resemble-ai/chatterbox.git"
repo_name = "chatterbox"

# Check if repo already exists
if os.path.exists(repo_name):
    print(f"Repository {repo_name} already exists. Skipping clone.")
else:
    print(f"Cloning repository from {repo_url}...")
    !git clone $repo_url
    print("Repository cloned successfully.")

# Change to repository directory
os.chdir(repo_name)
print(f"Changed to repository directory: {os.getcwd()}")

# Sync configuration to src/config.py
# Note: We exclude source_path and dest_path as they are Colab-specific
print("Syncing configuration to src/config.py...")

config_content = '''from dataclasses import dataclass, field
from typing import List, Literal, Optional
import os
import glob


def should_run_preprocessing(config) -> bool:
    """
    Determine if preprocessing should run based on config.preprocess setting.
    
    - If preprocess is True: always run
    - If preprocess is False: skip
    - If preprocess is "auto": check if preprocessed_dir exists and has same number of .pt files as .wav files
    
    Returns True if preprocessing should run, False otherwise.
    """
    if config.preprocess is True:
        return True
    elif config.preprocess is False:
        return False
    elif config.preprocess == "auto":
        # Check if preprocessed_dir exists
        if not os.path.exists(config.preprocessed_dir):
            return True
        
        # Count .wav files in wav_dir
        wav_files = glob.glob(os.path.join(config.wav_dir, "*.wav"))
        wav_count = len(wav_files)
        
        # Count .pt files in preprocessed_dir
        pt_files = glob.glob(os.path.join(config.preprocessed_dir, "*.pt"))
        pt_count = len(pt_files)
        
        # If counts don't match, rerun preprocessing
        if wav_count != pt_count:
            return True
        
        return False
    
    # Default to running preprocessing
    return True


@dataclass
class TrainConfig:
    # --- Paths ---
    # Directory where setup.py downloaded the files
    model_dir: str = "''' + model_dir + '''"
    
    # Path to your metadata CSV (Format: ID|RawText|NormText)
    csv_path: str = "''' + csv_path + '''"
    #metadata_path: str = "./metadata.json"
    
    # Directory containing WAV files
    wav_dir: str = "''' + wav_dir + '''"
    #wav_dir: str = "./FileBasedDataset"
    
    preprocessed_dir = "''' + preprocessed_dir + '''"
    #preprocessed_dir = "./FileBasedDataset/preprocess"
    
    # Output directory for the finetuned model
    output_dir: str = "''' + output_dir + '''"
    
    is_inference = ''' + str(is_inference).lower() + '''
    inference_prompt_path: str = "''' + inference_prompt_path + '''"
    inference_test_text: str = "''' + inference_test_text + '''"


    ljspeech = ''' + str(ljspeech).lower() + ''' # Set True if the dataset format is ljspeech, and False if it's file-based.
    json_format = ''' + str(json_format).lower() + ''' # Set True if the dataset format is json, and False if it's file-based or ljspeech.
    # Preprocessing mode: True (always run), False (skip), "auto" (smart detection)
    preprocess: Optional[Literal[True, False, "auto"]] = ''' + (f'"{preprocess}"' if isinstance(preprocess, str) else str(preprocess).lower()) + '''
    
    is_turbo: bool = ''' + str(is_turbo).lower() + '''  # Set True if you're training Turbo, False if you're training Normal.
    is_lora: bool = ''' + str(is_lora).lower() + '''   # True: Efficient LoRA training (Recommended for < 10h data)
                           # False: Full Fine-Tune (High VRAM, for massive datasets)
    is_merge_lora: bool = ''' + str(is_merge_lora).lower() + '''  # If True and is_lora is True, automatically run merge_lora.py after training

    lora_r: int = ''' + str(lora_r) + '''
    lora_alpha: int = ''' + str(lora_alpha) + '''
    turbo_lora_target_modules: List[str] = field(default_factory=lambda: ''' + str(turbo_lora_target_modules) + ''')
    lora_target_modules: List[str] = field(default_factory=lambda: ''' + str(lora_target_modules) + ''')
    lora_modules_to_save: List[str] = field(default_factory=lambda: ''' + str(lora_modules_to_save) + ''')
    

    # --- Vocabulary ---
    # The size of the NEW vocabulary (from tokenizer.json)
    # Ensure this matches the JSON file generated by your tokenizer script.
    # For Turbo mode: Use the exact number provided by setup.py (e.g., 52260)
    new_vocab_size: int = ''' + str(new_vocab_size) + ''' if is_turbo else 2454 

    # --- Hyperparameters ---
    batch_size: int = ''' + str(batch_size) + '''      # Adjust based on VRAM (2, 4, 8)
    grad_accum: int = ''' + str(grad_accum) + '''       # Effective Batch Size = Batch * Accum
    learning_rate: float = ''' + str(learning_rate) + ''' if is_lora else 1e-5  # T3 is sensitive, keep low
    num_epochs: int = ''' + str(num_epochs) + ''' if is_lora else 30
    
    save_steps: int = ''' + str(save_steps) + '''
    save_total_limit: int = ''' + str(save_total_limit) + '''
    dataloader_num_workers: int = ''' + str(dataloader_num_workers) + '''

    # --- Constraints ---
    start_text_token = 255
    stop_text_token = 0
    max_text_len: int = ''' + str(max_text_len) + '''
    max_speech_len: int = ''' + str(max_speech_len) + '''   # Truncates very long audio
    prompt_duration: float = ''' + str(prompt_duration) + ''' # Duration for the reference prompt (seconds)
'''

# Write the updated config file
with open('src/config.py', 'w') as f:
    f.write(config_content)

print("Configuration synced successfully to src/config.py")
print("\nUpdated configuration values:")
print(f"  model_dir: {model_dir}")
print(f"  csv_path: {csv_path}")
print(f"  wav_dir: {wav_dir}")
print(f"  preprocessed_dir: {preprocessed_dir}")
print(f"  output_dir: {output_dir}")
print(f"  is_turbo: {is_turbo}")
print(f"  is_lora: {is_lora}")
print(f"  is_merge_lora: {is_merge_lora}")
print(f"  lora_r: {lora_r}")
print(f"  lora_alpha: {lora_alpha}")
print(f"  batch_size: {batch_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  num_epochs: {num_epochs}")

## STEP 3: Google Drive Connection & Data Copy

Mount Google Drive and copy dataset from source_path to the Colab working directory.

In [ ]:
# STEP 3: Google Drive Connection & Copy Data

from google.colab import drive
import shutil
import os

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Create destination directory if it doesn't exist
os.makedirs(dest_path, exist_ok=True)

# Change to destination directory
os.chdir(dest_path)
print(f"Changed working directory to: {dest_path}")

# Copy dataset from Google Drive to Colab
print(f"Copying dataset from {source_path} to {dest_path}...")

# Check if source exists
if os.path.exists(source_path):
    # Copy the entire MyTTSDataset folder
    dest_dataset_path = os.path.join(dest_path, 'MyTTSDataset')
    if os.path.exists(dest_dataset_path):
        shutil.rmtree(dest_dataset_path)
    shutil.copytree(source_path, dest_dataset_path)
    print(f"Dataset copied successfully to {dest_dataset_path}")
else:
    print(f"Warning: Source path {source_path} does not exist. Please check your Google Drive.")

# Verify the copy
print("\nVerifying copied files:")
if os.path.exists(dest_dataset_path):
    for item in os.listdir(dest_dataset_path):
        item_path = os.path.join(dest_dataset_path, item)
        if os.path.isdir(item_path):
            count = len(os.listdir(item_path))
            print(f"  {item}/ ({count} items)")
        else:
            print(f"  {item}")

## STEP 4: Install Dependencies

Install system dependencies (ffmpeg) and Python requirements.

In [ ]:
# STEP 4: Install Dependencies

import os

# Update apt and install ffmpeg
print("Installing system dependencies (ffmpeg)...")
!sudo apt update
!sudo apt install -y ffmpeg

# Install Python requirements
print("\nInstalling Python requirements...")
!pip install -r requirements.txt

print("\nDependencies installed successfully.")

## STEP 5: Run Setup

Run the setup script to prepare pretrained models and other necessary components.

In [ ]:
# STEP 5: Run Setup

import os

print("Running setup.py...")
print(f"Working directory: {os.getcwd()}")

# Run setup.py
!python setup.py

print("\nSetup completed successfully.")

## STEP 6: Training & LoRA Merge

Run the training script. If `is_merge_lora` is True and `is_lora` is True, automatically merge LoRA weights after training.

In [ ]:
# STEP 6: Training & LoRA Merge

import os

print("Starting training...")
print(f"Working directory: {os.getcwd()}")
print(f"is_lora: {is_lora}")
print(f"is_merge_lora: {is_merge_lora}")

# Run training
!python train.py

# Check if we should merge LoRA
if is_lora and is_merge_lora:
    print("\n" + "="*50)
    print("LoRA training completed. Merging LoRA weights...")
    print("="*50 + "\n")
    !python merge_lora.py
    print("\nLoRA weights merged successfully.")
else:
    print("\nSkipping LoRA merge (is_lora=False or is_merge_lora=False).")

print("\n" + "="*50)
print("Training pipeline completed!")
print("="*50)

## Post-Training: Copy Results to Google Drive

Optional: Copy the trained model output back to Google Drive for safekeeping.

In [ ]:
# Optional: Copy trained model back to Google Drive

import shutil
import os

# Define output directory and Google Drive destination
output_dir_local = "./chatterbox_output"
output_dir_gdrive = "/content/drive/MyDrive/chatterbox_output"

if os.path.exists(output_dir_local):
    print(f"Copying trained model from {output_dir_local} to {output_dir_gdrive}...")
    
    # Remove existing directory in Google Drive if it exists
    if os.path.exists(output_dir_gdrive):
        shutil.rmtree(output_dir_gdrive)
    
    # Copy the output directory
    shutil.copytree(output_dir_local, output_dir_gdrive)
    print(f"Trained model copied successfully to {output_dir_gdrive}")
else:
    print(f"Output directory {output_dir_local} does not exist. Skipping copy.")